# specset usecase 2

**Create a Python environment**

venv at /d0/tp/specset_experiments/venv_peptdeep151/
. /d0/tp/specset_experiments/venv_peptdeep151/bin/activate

**Install alphapeptdeep, alphabase**
pip install "alphabase[full-stable]"  # important for saving HDF library. TypeError: 
pip install "alphabase[full-stable]"
pip install "peptdeep[stable,gui-stable]"

Strings must be encoded before hashing

**Download pretrained models**
https://github.com/MannLabs/alphapeptdeep/releases/download/pre-trained-models/pretrained_models_v3.zip

**Install specset**

pip install -e ~/aiproteomics/dbl_alphapepdeep

In [ ]:
from pathlib import Path

# change path to the data here
p = Path("/data/scratch/specset_data/hela_dda")

from specset.utils import maxquant_to_specset

print("Create specset from MaxQuant search result...")
maxquant_to_specset(str(p / 'txt'), str(p / 'txt.specset'), additional_columns = {"Score" : "score", "Proteins" : "proteins", "Gene Names" : "genes"})


# To AI data

In [ ]:
#pip install "alphabase[full-stable]"

import os
os.environ["TQDM_DISABLE"] = "1"                  # Suppress tqdm progress bar

import tqdm

from specset.utils import specset_to_alphapeptdeep_string

print("Create alphapeptdeep stream...")
str_stream = specset_to_alphapeptdeep_string(str(p / 'txt.specset'))

dfs = []
frag_inten_dfs = []
from alphabase.spectral_library.reader import LibraryReaderBase
_lib = LibraryReaderBase()

n = 0
for psm in str_stream:    
    n += 1
    if n % 1000 == 0:
        print(f"n = {n}", end = '\r')
    a = _lib.import_file(psm)
    dfs.append(a)
    frag_inten_dfs.append(_lib.fragment_intensity_df)

print(f"Number of PSMs = {n}")

import pickle
f = open(p / 'txt.specset.dfs.pckl', 'wb')
pickle.dump(dfs, f)
f.close()

f = open(p / 'txt.specset.frag_inten_dfs.pckl', 'wb')
pickle.dump(frag_inten_dfs, f)
f.close()

Create specset from MaxQuant search result...
Reading msms.txt...


KeyboardInterrupt: 

# Refine the generic MS2 model

In [ ]:
#pip install "alphabase[full-stable]"
#pip install "peptdeep[stable,gui-stable]"


import torch # noqa: 401, to prevent crash in Mac Arm
import os
import warnings

import pickle

print('Load PSMs...')
with open(os.path.expanduser('~/scratch/hela_dda/txt.specset.dfs.pckl'), 'rb') as fp:
    dfs = pickle.load(fp)
with open(os.path.expanduser('~/scratch/hela_dda/txt.specset.frag_inten_dfs.pckl'), 'rb') as fp:
    frag_inten_dfs = pickle.load(fp)

print('Prepare alphapeptdeep data...')
from alphabase.peptide.fragment import concat_precursor_fragment_dataframes

psm_df, frag_df = concat_precursor_fragment_dataframes(dfs, frag_inten_dfs)

# fine tune
from peptdeep.model.ms2 import pDeepModel

model_path = os.path.expanduser("~/scratch/specset_experiments/generic/ms2.pth")
model_interface = pDeepModel(override_from_weights=True)
model_interface.load(model_path)

supported_fragment_types = model_interface.model.supported_charged_frag_types
print(f"Fragment types supported by the model: {supported_fragment_types}")

psm_df['nce'] = 30.0
psm_df['instrument'] = 'timsTOF'

print('MS2 model adaptation...')

model_interface.train(precursor_df=psm_df, fragment_intensity_df=frag_df.loc[:, supported_fragment_types], 
                      lr= 0.0001,
                      batch_size=512,
                      warmup_epoch=10, epoch=20, verbose=1)

#save model
model_interface.save(os.path.expanduser('~/scratch/specset_experiments/hela_dda/transfer_ms2_hela/transfer_ms2.pth'))


#####
print('Generate library using adapted MS2 model...')

from peptdeep.pipeline_api import global_settings, generate_library

fasta = os.path.expanduser('~/scratch/specset_experiments/uniprot_Human.fasta')

lib_settings = global_settings['library']

lib_settings['infile_type'] = 'fasta'
lib_settings['infiles'] = [fasta]
lib_settings['fasta']['add_contaminants'] = False
lib_settings['output_tsv']['enabled'] = True
lib_settings['output_folder'] = os.path.expanduser('~/scratch/specset_experiments/hela_dda/lib_transfer_ms2_hela')


mgr_settings = global_settings['model_mgr']

mgr_settings['external_ms2_model'] = os.path.expanduser('~/scratch/specset_experiments/hela_dda/transfer_ms2_hela/transfer_ms2.pth')
mgr_settings['external_rt_model'] = os.path.expanduser('~/scratch/specset_experiments/generic/rt.pth')
mgr_settings['external_ccs_model'] = os.path.expanduser('~/scratch/specset_experiments/generic/ccs.pth')

generate_library()

#####
print('Generate library using the default model...')
lib_settings['output_folder'] = os.path.expanduser('~/scratch/specset_experiments/hela_dda/lib_default_hela')

mgr_settings['external_ms2_model'] = os.path.expanduser('~/scratch/specset_experiments/generic/ms2.pth')
mgr_settings['external_rt_model'] = os.path.expanduser('~/scratch/specset_experiments/generic/rt.pth')
mgr_settings['external_ccs_model'] = os.path.expanduser('~/scratch/specset_experiments/generic/ccs.pth')

generate_library()


Load PSMs...
Prepare alphapeptdeep data...


/d0/tp/specset_experiments/venv_peptdeep151/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fragment types supported by the model: ['b_z1', 'b_z2', 'y_z1', 'y_z2', 'b_modloss_z1', 'b_modloss_z2', 'y_modloss_z1', 'y_modloss_z2']
MS2 model adaptation...
2026-09-06 06:56:56> Training with fixed sequence length: 0


/d0/tp/specset_experiments/venv_peptdeep151/lib/python3.10/site-packages/peptdeep/model/ms2.py:733: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  to_be_loaded_state_dict = t

[Training] Epoch=1, lr=2e-05, loss=0.012505643252839663
[Training] Epoch=2, lr=3e-05, loss=0.011711263119829344
[Training] Epoch=3, lr=4e-05, loss=0.01159476182495714
[Training] Epoch=4, lr=5e-05, loss=0.011511054305475917
[Training] Epoch=5, lr=6e-05, loss=0.011444638418966802
[Training] Epoch=6, lr=7e-05, loss=0.011432255094955431
[Training] Epoch=7, lr=8e-05, loss=0.011532139331049554
[Training] Epoch=8, lr=9e-05, loss=0.011403805171341999
[Training] Epoch=9, lr=0.0001, loss=0.011377508634092226
[Training] Epoch=10, lr=0.0001, loss=0.011358218977935541
[Training] Epoch=11, lr=9.755282581475769e-05, loss=0.011333665247154484
[Training] Epoch=12, lr=9.045084971874738e-05, loss=0.011273535929747024
[Training] Epoch=13, lr=7.938926261462366e-05, loss=0.011226244361909188
[Training] Epoch=14, lr=6.545084971874738e-05, loss=0.011180483326773618
[Training] Epoch=15, lr=5e-05, loss=0.011141649171417765
[Training] Epoch=16, lr=3.4549150281252636e-05, loss=0.011074651769469783
[Training] Epoc

...

2026-09-06 07:35:55> Using external ms2 model: '/d0/tp/specset_experiments/generic/ms2.pth'


/d0/tp/specset_experiments/venv_peptdeep151/lib/python3.10/site-packages/peptdeep/model/model_interface.py:738: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(stre

2026-09-06 07:35:55> Using external rt model: '/d0/tp/specset_experiments/generic/rt.pth'
2026-09-06 07:35:55> Using external ccs model: '/d0/tp/specset_experiments/generic/ccs.pth'
2026-09-06 07:35:55> xxx/library.tsv does not exist, use default IRT_PEPTIDE_DF to translate irt
2026-09-06 07:35:55> Generating the spectral library ...


/d0/tp/specset_experiments/venv_peptdeep151/lib/python3.10/site-packages/peptdeep/model/model_interface.py:738: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(stre

Predicting charge states for 3447252 peptides ...
2026-09-06 07:38:55> Loaded 4109724 precursors.
2026-09-06 07:39:09> Predicting RT/IM/MS2 for 4109723 precursors ...
2026-09-06 07:39:09> Predicting RT ...
2026-09-06 07:40:12> Predicting mobility ...
2026-09-06 07:42:00> Predicting MS2 ...
2026-09-06 07:44:32> End predicting RT/IM/MS2
2026-09-06 07:44:32> Predicting the spectral library with 4109723 precursors and 287.66M fragments used 18.0913 GB memory
2026-09-06 07:44:32> Saving HDF library to /d0/tp/specset_experiments/hela_dda/lib_default_hela/predict.speclib.hdf ...
2026-09-06 07:45:02> Translating to /d0/tp/specset_experiments/hela_dda/lib_default_hela/predict.speclib.tsv for DiaNN/Spectronaut...
Translation finished, it will take several minutes to export the rest precursors to the tsv file...
2026-09-06 08:09:04> Library generated!!
